# 🌍 Análisis de Islas Galápagos con Google Earth Engine
**Teledetección y Visión por Computadora**  
Objetivo: Adquisición, pre-procesamiento, visualización y lógica inicial sobre imágenes satelitales de Galápagos.

---
## 1. 🛰️ Adquisición — Conexión a GEE y descarga de ImageCollection

In [ ]:
import ee
import geemap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Autenticar e inicializar (reemplaza con tu project ID)
# ee.Authenticate()  # Solo necesario la primera vez
ee.Initialize(project='TU_PROJECT_ID')

print('✅ Conexión con Google Earth Engine exitosa')

In [ ]:
# ── Definir área de interés: Islas Galápagos ──────────────────────────────────
galapagos = ee.Geometry.Rectangle([-92.0, -1.5, -89.0, 0.8])

# ── Cargar ImageCollection: Sentinel-2 SR (Level-2A) ─────────────────────────
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(galapagos)
    .filterDate('2023-01-01', '2023-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # < 20% nubes
    .sort('system:time_start')
)

print(f'📦 Imágenes encontradas: {collection.size().getInfo()}')

---
## 2. ⚙️ Pre-procesamiento — Clipping y Remoción de Nubes

In [ ]:
# ── Función para enmascarar nubes con SCL (Scene Classification Layer) ────────
def mask_clouds_s2(image):
    """Remueve nubes y sombras usando la banda SCL de Sentinel-2."""
    scl = image.select('SCL')
    # SCL: 3=sombra de nube, 8=nube media prob, 9=nube alta prob, 10=cirrus
    cloud_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(cloud_mask)

# ── Función para recortar al área de Galápagos ────────────────────────────────
def clip_to_galapagos(image):
    return image.clip(galapagos)

# ── Aplicar pre-procesamiento a toda la colección ─────────────────────────────
processed = (
    collection
    .map(mask_clouds_s2)
    .map(clip_to_galapagos)
)

print('✅ Pre-procesamiento aplicado: clipping + remoción de nubes')
print(f'📦 Imágenes procesadas: {processed.size().getInfo()}')

---
## 3. 🖼️ Visualización — 10 Imágenes (Fechas y Bandas)

In [ ]:
# ── Seleccionar 10 imágenes representativas ───────────────────────────────────
images_list = processed.toList(10)

# Parámetros de visualización RGB (bandas B4=Rojo, B3=Verde, B2=Azul)
vis_rgb = {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']}
# Parámetros NIR (B8=Infrarrojo Cercano, B4=Rojo, B3=Verde)
vis_nir = {'min': 0, 'max': 4000, 'bands': ['B8', 'B4', 'B3']}

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('🌍 Galápagos — 10 Imágenes Sentinel-2 (2023)', fontsize=16, fontweight='bold')

for i in range(10):
    ax = axes[i // 5][i % 5]
    
    img = ee.Image(images_list.get(i))
    date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    
    # Alternar entre RGB y NIR para mostrar diferentes bandas
    vis_params = vis_rgb if i % 2 == 0 else vis_nir
    banda_tipo = 'RGB' if i % 2 == 0 else 'NIR'
    
    # Obtener thumbnail
    thumb_url = img.getThumbURL({
        **vis_params,
        'region': galapagos,
        'dimensions': 256,
        'format': 'png'
    })
    
    # Descargar y mostrar
    import requests
    from PIL import Image
    from io import BytesIO
    
    response = requests.get(thumb_url)
    img_pil = Image.open(BytesIO(response.content))
    
    ax.imshow(img_pil)
    ax.set_title(f'{date}\n[{banda_tipo}]', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('galapagos_10_imagenes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualización de 10 imágenes completada')

---
## 4. 🧮 Lógica Inicial — NDVI + Segmentación Agua/Tierra

In [ ]:
# ── Crear imagen compuesta (mediana anual) ────────────────────────────────────
composite = processed.median().clip(galapagos)

# ── Calcular NDVI ─────────────────────────────────────────────────────────────
# NDVI = (NIR - Rojo) / (NIR + Rojo)  →  Banda B8 (NIR) y B4 (Rojo)
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')

# ── Calcular NDWI (Índice de Agua) ────────────────────────────────────────────
# NDWI = (Verde - NIR) / (Verde + NIR)  →  detecta agua
ndwi = composite.normalizedDifference(['B3', 'B8']).rename('NDWI')

print('✅ NDVI y NDWI calculados sobre imagen compuesta mediana')

In [ ]:
# ── Segmentación: Agua / Tierra / Vegetación ──────────────────────────────────
# Umbrales:
#   NDWI > 0.1  → Agua
#   NDVI > 0.3  → Vegetación
#   resto       → Suelo desnudo / lava

agua      = ndwi.gt(0.1)               # 1 = agua
vegetacion = ndvi.gt(0.3).And(agua.Not())  # 1 = vegetación (no agua)
suelo     = agua.Not().And(vegetacion.Not())  # 1 = suelo/lava

# Combinar en una sola imagen de clases: 0=suelo, 1=vegetación, 2=agua
clasificacion = (
    suelo.multiply(0)
    .add(vegetacion.multiply(1))
    .add(agua.multiply(2))
    .rename('clasificacion')
)

print('✅ Segmentación aplicada: Agua / Vegetación / Suelo')

In [ ]:
# ── Visualizar NDVI, NDWI y Clasificación ─────────────────────────────────────
import requests
from PIL import Image
from io import BytesIO

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle('🌿 Galápagos — NDVI, NDWI y Segmentación', fontsize=15, fontweight='bold')

imagenes = [
    (composite, {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']}, 'RGB Compuesto'),
    (ndvi,      {'min': -0.2, 'max': 0.8, 'palette': ['#d73027','#fee08b','#1a9850']}, 'NDVI'),
    (ndwi,      {'min': -0.5, 'max': 0.5, 'palette': ['#8c510a','#f5f5f5','#01665e']}, 'NDWI'),
    (clasificacion, {'min': 0, 'max': 2, 'palette': ['#b5651d','#228B22','#1E90FF']}, 'Segmentación\nSuelo/Veg/Agua'),
]

for ax, (img_ee, vis, titulo) in zip(axes, imagenes):
    url = img_ee.getThumbURL({
        **vis,
        'region': galapagos,
        'dimensions': 512,
        'format': 'png'
    })
    resp = requests.get(url)
    img_pil = Image.open(BytesIO(resp.content))
    ax.imshow(img_pil)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.axis('off')

# Leyenda para clasificación
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#b5651d', label='Suelo / Lava'),
    Patch(facecolor='#228B22', label='Vegetación'),
    Patch(facecolor='#1E90FF', label='Agua'),
]
axes[3].legend(handles=legend_elements, loc='lower left', fontsize=9,
               framealpha=0.8, fancybox=True)

plt.tight_layout()
plt.savefig('galapagos_ndvi_segmentacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualización de índices y segmentación completada')

In [ ]:
# ── Estadísticas de cobertura ─────────────────────────────────────────────────
stats_ndvi = ndvi.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
    geometry=galapagos,
    scale=1000,
    maxPixels=1e9
).getInfo()

print('📊 Estadísticas NDVI en Galápagos (2023):')
print(f"   Media : {stats_ndvi.get('NDVI_mean',  'N/A'):.4f}")
print(f"   Mínimo: {stats_ndvi.get('NDVI_min',   'N/A'):.4f}")
print(f"   Máximo: {stats_ndvi.get('NDVI_max',   'N/A'):.4f}")
print()
print('🌿 NDVI > 0.3  → Vegetación densa')
print('🏜️  NDVI < 0.1  → Suelo desnudo / lava volcánica')
print('💧 NDWI > 0.1  → Presencia de agua')

---
## 5. 🗺️ Mapa Interactivo (Bonus)
Usando `geemap` para visualización interactiva.

In [ ]:
# ── Mapa interactivo con geemap ───────────────────────────────────────────────
Map = geemap.Map(center=[-0.5, -90.5], zoom=8)

Map.addLayer(composite,     {'min': 0, 'max': 3000, 'bands': ['B4','B3','B2']}, 'RGB Compuesto')
Map.addLayer(composite,     {'min': 0, 'max': 4000, 'bands': ['B8','B4','B3']}, 'NIR Falso Color')
Map.addLayer(ndvi,          {'min': -0.2, 'max': 0.8, 'palette': ['#d73027','#fee08b','#1a9850']}, 'NDVI')
Map.addLayer(clasificacion, {'min': 0, 'max': 2,   'palette': ['#b5651d','#228B22','#1E90FF']}, 'Segmentación')

Map.addLayerControl()
Map

---
## ✅ Resumen de la Entrega

| Requisito | Implementación |
|-----------|----------------|
| **1. Adquisición** | Sentinel-2 SR sobre Galápagos (2023), filtrado por nubosidad < 20% |
| **2. Pre-procesamiento** | Clipping al bbox de Galápagos + máscara de nubes via SCL |
| **3. Visualización** | 10 imágenes (RGB y NIR) de diferentes fechas |
| **4. Lógica Inicial** | NDVI (vegetación), NDWI (agua) + segmentación de 3 clases |
| **Bonus** | Mapa interactivo con geemap + estadísticas descriptivas |